In [1]:
!pip install ml_collections

In [1]:
import os
import sys
import shutil
import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F


from tqdm import tqdm, trange

import warnings


from ml_collections import ConfigDict

In [2]:
sys.path.append('..')

In [3]:
from CRT_utils.CRT.core.model1 import Model
# from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.littlehelper import *

# from CRT_utils.CRT.test import test
from CRT_utils.CRT.test_zero_context_zero_target import test

# from CRT_utils.CRT.test_visual_search import test
# from utils.evaluate_uncertainty import evaluate_uncertainty
from CRT_utils.CRT.core.config import create_config, save_config
from CRT_utils.CRT.core.dataset import COCODataset, COCODatasetWithID, COCODatasetMixOR, COCODatasetFullMix
from CRT_utils.CRT.core.model1 import Model
# from CRT_utils.CRT.core.metrics import AccuracyLogger 


# User define Variables (Arguments)

In [4]:
config_dict = ConfigDict()


config_dict['config']                = None
config_dict['outdir']                = '../CRT_utils/CRT_weights_and_config/CRT_COCO_random/CRT no bbox PE new json/'
config_dict['checkpoint']            = None # '../CRT_utils/CRT_weights_and_config/CRT_COCO_random/Archive 1/checkpoint_30.tar'

config_dict['annotations']           = '../datasets/COCO18_dset_for_CRT_training/coco18_newtrain.json'
config_dict['imagedir']              = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt'

config_dict['test_annotations']      = '../datasets/COCO18_dset_for_CRT_training/coco18_newtest_cleaned.json'
config_dict['test_imagedir']         = '../datasets/COCO18_dset_for_CRT_training/coco18_for_crt'
config_dict['test_frequency']        = 1

config_dict['epochs']                = 50
config_dict['save_frequency']        = 1
config_dict['print_batch_metrics']   = None

config_dict['batch_size']            = 32 
config_dict['learning_rate']         = None
config_dict['imbalance_reweighting'] = None
config_dict['num_decoder_heads']     = None
config_dict['num_decoder_layers']    = 6
config_dict['uncertainty_gate_type'] = None
config_dict['uncertainty_threshold'] = 0
config_dict['weighted_prediction']   = None

In [5]:
if not os.path.exists(config_dict['outdir']):
    os.makedirs(config_dict['outdir'])

## Setting up category_idx_dict.pkl for image augmentation

In [6]:
import json
import pickle

with open(config_dict['annotations'], 'rb') as file:
    train_metadata = json.load(file)
    
categories_id_to_name = {}

# _____ CREATE A LOOKUP TABLE FOR CATEGORY ID TO CATEGORY NAME _____
for info in train_metadata['categories']:
    categories_id_to_name[info['id']] = info['name']
# _____ CREATE A LOOKUP TABLE FOR CATEGORY ID TO CATEGORY NAME _____
    
    
    
    
    
# _____ CREATE A LOOKUP TABLE FOR INDEXES IN THE SAME CATEGORIES _____
category_idx_dict = {}

for i in range(len(train_metadata['annotations'])):
    
    info = train_metadata['annotations'][i]
    
    name = categories_id_to_name[info['category_id']]
    
    if name not in category_idx_dict:
        category_idx_dict[name] = []
        
    category_idx_dict[name].append(i)
# _____ CREATE A LOOKUP TABLE FOR INDEXES IN THE SAME CATEGORIES _____



with open('../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl', 'wb') as file:
    pickle.dump(category_idx_dict, file, protocol=pickle.HIGHEST_PROTOCOL)

# Setting configure variables

In [7]:
cfg = create_config(config_dict)

In [8]:
dataset = COCODatasetFullMix(
    cfg.annotations, 
    cfg.imagedir, 
    image_size      = (224,224), 
    # _____ ORIGINAL CODE _____
    normalize_means = [0.485, 0.456, 0.406], 
    normalize_stds  = [0.229, 0.224, 0.225],
    # _____ ORIGINAL CODE _____
    
    # _____ MODIFIED VERSION _____
#     normalize_means = [0.5, 0.5, 0.5], 
#     normalize_stds  = [0.5, 5, 0.5]
    # _____ MODIFIED VERSION _____
    
    # _____ ADDED CODE _____
    category_dic_dir = '../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl'
    
    # _____ ADDED CODE _____
    
)

dataloader = DataLoader(
    dataset, 
    batch_size  = cfg.batch_size, 
    num_workers = 1, 
    shuffle     = True, 
    pin_memory  = True, 
    drop_last   = True
)


-------------------------------
Annotation Counts
-------------------------------
potted plant               8631
tv                         5803
bottle                    24070
chair                     38073
car                       43533
stop sign                  1983
clock                      6320
cup                       20574
fork                       5474
knife                      7760
bowl                      14323
toilet                     4149
laptop                     4960
mouse                      2261
keyboard                   2854
microwave                  1672
oven                       3334
sink                       5609
Total                    201383
-------------------------------



In [9]:
NUM_CLASSES     = dataset.NUM_CLASSES
cfg.num_classes = NUM_CLASSES

os.makedirs(config_dict.outdir, exist_ok=True)

save_config(cfg, config_dict.outdir)

print(cfg)

annotations: ../datasets/COCO18_dset_for_CRT_training/coco18_newtrain.json
batch_size: 32
checkpoint: null
git: 1c647634b6de3d66ab4a5a840cd3721dc41743ab
imagedir: ../datasets/COCO18_dset_for_CRT_training/coco18_for_crt
imbalance_reweighting: false
learning_rate: 1.0e-05
num_classes: 18
num_decoder_heads: 8
num_decoder_layers: 6
test_annotations: ../datasets/COCO18_dset_for_CRT_training/coco18_newtest_cleaned.json
test_imagedir: ../datasets/COCO18_dset_for_CRT_training/coco18_for_crt
uncertainty_gate_type: learned
uncertainty_threshold: 0
weighted_prediction: false



In [10]:
model = Model.from_config(cfg)

In [11]:
assert(model.TARGET_IMAGE_SIZE == model.CONTEXT_IMAGE_SIZE == dataset.image_size), "Image size from the dataset is not compatible with the encoder."

In [12]:
device = (
    "cuda" if torch.cuda.is_available()
    else "mps"  # macbook uses metal performance shaders to GPU accelearation
    if torch.backends.mps.is_available()
    else "cpu"
)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay = 0.01)





# _____ ORIGINAL CODE _____

# if cfg.imbalance_reweighting:
#     class_weights = torch.true_divide(dataset.relative_annotation_counts.max(), dataset.relative_annotation_counts)
#     criterion     = nn.CrossEntropyLoss(weight= class_weights.to(device))
# else:
#     criterion = nn.CrossEntropyLoss()
    
# _____ ORIGINAL CODE _____


# _____ MODIFIED CODE _____

criterion = nn.CrossEntropyLoss()

# _____ MODIFIED CODE _____


    
    
    

if cfg.checkpoint is not None:
    
    print("Initializing from checkpoint {}".format(cfg.checkpoint))
    
    checkpoint = torch.load(cfg.checkpoint, map_location="cpu")
    
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
else:
    print("No checkpoint was passed.")
    
    model.to(device)
    start_epoch = 1

# Tensorboard
writer = SummaryWriter(log_dir=os.path.join(config_dict.outdir, "runs/{date:%Y-%m-%d_%H%M}".format(date=datetime.datetime.now())))

context_images, target_images, bbox, labels = next(iter(dataloader))

writer.add_images("context_image_batch", context_images) # add example context image batch to tensorboard log
writer.add_images("target_image_batch", target_images)   # add example target image batch to tensorboard log

with warnings.catch_warnings(): # add_graph method is known to issue a warning
    warnings.simplefilter("ignore")
    
    # _____ ORIGINAL CODE _____
#     writer.add_graph(model, input_to_model=[context_images.to(device), target_images.to(device), bbox.to(device)]) # add model graph to tensorboard log
    # _____ ORIGINAL CODE _____
    
    
    # _____ MODIFIED VERSION _____
    
    writer.add_graph(
        model, 
        input_to_model=[
            context_images.to(device), 
            target_images.to(device)
        ]
    ) # add model graph to tensorboard log
    
    # _____ MODIFIED VERSION _____
    
    
# _____ ORIGINAL CODE _____
# accuracy_logger_main_branch = AccuracyLogger(dataset.idx2label)
# _____ ORIGINAL CODE _____



# _____ MODIFIED CODE _____
# accuracy_logger_main_branch = AccuracyLogger(num_classes=49)
# _____ MODIFIED CODE _____

No checkpoint was passed.


/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


# Training

In [13]:
## _____ ORIGINAL CODE _____
# for epoch in tqdm(range(start_epoch, config_dict.epochs + 1), position=0, desc="Epochs", leave=True):

# model.extended_output = True

temperature = 0.1


for epoch in range(start_epoch, config_dict.epochs + 1):

    print(f'Epoch {epoch}/{config_dict.epochs}:')
    model.train() # set train mode
#     accuracy_logger_main_branch.reset() # reset accuracy logger every epoch
    # accuracy_logger_uncertainty_branch.reset()

    for i, (context_images, target_images, bbox, labels_cpu) in enumerate(tqdm(dataloader, position=0, desc="Batches", leave=True)):
        
        context_images = context_images.to(device)
        target_images  = target_images.to(device)
        
        bbox   = bbox.to(device)
        labels = labels_cpu.to(device) # keep a copy of labels on cpu to avoid unnecessary transfer back to cpu later

        # output_uncertainty_branch , output_main_branch, output_weighted, uncertainty = model(context_images, target_images, bbox)
        
        
        # _____ ORIGINAL CODE _____
#         output_main_branch = model(context_images, target_images, bbox)
        # _____ ORIGINAL CODE _____
    
    
    
        # _____ MODIFIED VERSION _____
        output_main_branch, attention_map = model(context_images, target_images)
        
        attention_map = attention_map[:, cfg.num_decoder_layers - 1, 0]
        
        
        # _____ MODIFIED VERSION _____
        

        # backpropagation through both branches
        optimizer.zero_grad(set_to_none=True)

        # if cfg.uncertainty_gate_type == "learned" or cfg.uncertainty_gate_type == "learned_metric":
        #     loss_uncertainty_estimator = criterion(output_weighted, labels)
        #     loss_uncertainty_estimator.backward(retain_graph=True)    

        # loss_uncertainty_branch = criterion(output_uncertainty_branch, labels)
        # loss_uncertainty_branch.backward(retain_graph=True)

#         print(attention_map.shape)
#         print(bbox2token(model, bbox).shape)
        
        # _____ MODIFIED CODE _____
        
        
        loss_main_branch = criterion(output_main_branch, labels)
        
#         loss_main_branch = criterion(
#             torch.log(attention_map + 1e-8), 
#             bbox2token(model, bbox)
#         )
        # _____ MODIFIED CODE _____
        
        

        
        loss_main_branch.backward()

        optimizer.step()
        
        # log metrics
        # _, predictions_uncertainty_branch = torch.max(output_uncertainty_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        # batch_accuracy_uncertainty_branch = sum(predictions_uncertainty_branch == labels_cpu) / cfg.batch_size
        # batch_loss_uncertainty_branch = loss_uncertainty_branch.item()
        # writer.add_scalar("Batch Accuracy Uncertainty Branch/train", batch_accuracy_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # writer.add_scalar("Batch Loss Uncertainty Branch/train", batch_loss_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # accuracy_logger_uncertainty_branch.update(predictions_uncertainty_branch, labels_cpu)

        
        
#         _, predictions_main_branch = torch.max(output_main_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        
        predictions_main_branch = torch.argmax(output_main_branch.detach().to("cpu"), 1)
        
        batch_accuracy_main_branch = sum(predictions_main_branch == labels.to('cpu')) / cfg.batch_size
        batch_loss_main_branch     = loss_main_branch.item()
        
        writer.add_scalar("Batch Accuracy Main Branch/train", batch_accuracy_main_branch, i + (epoch - 1) * len(dataloader))
        
        writer.add_scalar("Batch Loss Main Branch/train", batch_loss_main_branch, i + (epoch - 1) * len(dataloader))
#         accuracy_logger_main_branch.update(predictions_main_branch, bbox2token(model, bbox).to('cpu'))

        # writer.add_scalar("Batch Uncertainty/train", torch.mean(uncertainty), i + (epoch - 1) * len(dataloader))

        if config_dict.print_batch_metrics:
            print("\t Epoch {}, Batch {}: \t Loss: {} \t Accuracy: {}".format(epoch, i, batch_loss_main_branch, batch_accuracy_main_branch))


    # log metrics
#     writer.add_scalar("Total Accuracy Main Branch/train", accuracy_logger_main_branch.accuracy(), epoch * len(dataloader))
#     writer.add_scalar("Total Accuracy Uncertainty Branch/train", accuracy_logger_uncertainty_branch.accuracy(), epoch * len(dataloader))

#     print("\nEpoch {}, Train Accuracy: {}".format(epoch, accuracy_logger_main_branch.accuracy()))
#     print("{0:20} {1:10}".format("Class", "Accuracy")) # header
    
    
#     for name, acc in accuracy_logger_main_branch.named_class_accuarcies().items():
#         writer.add_scalar("Class Accuracies Main Branch/train/{}".format(name), acc, epoch * len(dataloader))
#         print("{0:20} {1:10.4f}".format(name, acc))

    # for name, acc in accuracy_logger_uncertainty_branch.named_class_accuarcies().items():
    #     writer.add_scalar("Class Accuracies Uncertainty Branch/train/{}".format(name), acc, epoch * len(dataloader))

    # save checkpoint and training accuracies
    if epoch % config_dict.save_frequency == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, config_dict.outdir + "/checkpoint_{}.tar".format(epoch))
        print("Checkpoint saved.")

#         accuracy_logger_main_branch.save(config_dict.outdir, name="train_accuracies_epoch_{}".format(epoch))
        # accuracy_logger_uncertainty_branch.save(args.outdir, name="train_accuracies_uncertainty_branch_epoch_{}".format(epoch))
    
    # evaluation on test data
    
    
    
    
    
    
    
    
    
    if cfg.test_annotations is not None and cfg.test_imagedir is not None and epoch % config_dict.test_frequency == 0:
        print("Starting evaluation on test data.")
        test_accuracy = test(
            model, 
            cfg.test_annotations, 
            cfg.test_imagedir, 
#             '../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl',
            outdir = config_dict.outdir,
            epoch  = epoch,
        )
        
        
        
        
        
        
        
        
        
        
        
        

        writer.add_scalar("Total Accuracy/test", test_accuracy.accuracy(), epoch * len(dataloader))
        
        for name, acc in test_accuracy.named_class_accuarcies().items():
            writer.add_scalar("Class Accuracies/test/{}".format(name), acc, epoch * len(dataloader))

        # print("Starting uncertainty evaluation.")
        # test_uncertainty_log = evaluate_uncertainty(model, cfg.test_annotations, cfg.test_imagedir)
        # writer.add_figure("Uncertainty Threshold Curve", test_uncertainty_log.plot_accuracy_vs_threshold(), epoch * len(dataloader))

        # if (args.epochs - epoch) / args.test_frequency < 1: # last evaluation
        #     writer.add_hparams({"learning_rate": cfg.learning_rate, "num_decoder_layers": cfg.num_decoder_layers, "num_decoder_heads": cfg.num_decoder_heads,
        #                         "uncertainty_gate_type": cfg.uncertainty_gate_type, "uncertainty_threshold": cfg.uncertainty_threshold, "imbalance_reweighting": str(cfg.imbalance_reweighting)},
        #                         metric_dict={"hparam/accuracy": test_accuracy.accuracy()})
        
writer.close()

Epoch 1/50:


Batches:   0%|                                         | 0/6293 [00:00<?, ?it/s]/Users/nguyentuan/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
Batches: 100%|████████████████████████████| 6293/6293 [1:13:12<00:00,  1.43it/s]


Checkpoint saved.
Starting evaluation on test data.
-------------------------------
Annotation Counts
-------------------------------
chair                       253
fork                        230
sink                        279
tv                          281
bowl                        141
car                         104
clock                       119
cup                         276
keyboard                    184
knife                       141
laptop                      123
mouse                       109
oven                        101
potted plant                154
toilet                      158
bottle                      166
stop sign                   126
microwave                   156
Total                      3101
-------------------------------



Test Batches: 100%|█████████████████████████| 3101/3101 [02:59<00:00, 17.23it/s]



Total Test Accuracy: 0.6250426769256592
Class                Accuracy  
car                      0.7981
stop sign                0.6429
bottle                   0.6867
cup                      0.5761
fork                     0.5391
knife                    0.2128
bowl                     0.4610
chair                    0.6522
potted plant             0.5844
toilet                   0.8734
tv                       0.7865
laptop                   0.4553
mouse                    0.8165
keyboard                 0.7609
microwave                0.4615
oven                     0.6238
sink                     0.6810
clock                    0.6387
Epoch 2/50:


...

Checkpoint saved.
Starting evaluation on test data.
-------------------------------
Annotation Counts
-------------------------------
chair                       253
fork                        230
sink                        279
tv                          281
bowl                        141
car                         104
clock                       119
cup                         276
keyboard                    184
knife                       141
laptop                      123
mouse                       109
oven                        101
potted plant                154
toilet                      158
bottle                      166
stop sign                   126
microwave                   156
Total                      3101
-------------------------------



Test Batches: 100%|█████████████████████████| 3101/3101 [03:04<00:00, 16.85it/s]



Total Test Accuracy: 0.8096120357513428
Class                Accuracy  
car                      0.8750
stop sign                0.9048
bottle                   0.9036
cup                      0.4783
fork                     0.8783
knife                    0.3830
bowl                     0.5674
chair                    0.8735
potted plant             0.6818
toilet                   0.9620
tv                       0.8754
laptop                   0.7805
mouse                    0.9633
keyboard                 0.9620
microwave                0.7692
oven                     0.8911
sink                     0.9247
clock                    0.8992
Epoch 29/50:


Batches:  22%|██████▋                       | 1391/6293 [16:28<58:03,  1.41it/s]


KeyboardInterrupt: 

In [ ]:
test_accuracy = test(
    model, 
    cfg.test_annotations, 
    cfg.test_imagedir, 
#             '../datasets/COCO18_dset_for_CRT_training/category_idx_dict.pkl',
    outdir = config_dict.outdir,
    epoch  = epoch,
)

In [ ]:
len(dataset)

In [ ]:
predictions_main_branch

In [16]:
bbox2token(model, bbox).to('cpu')

tensor([39, 28, 15, 14, 32, 18, 36, 33, 37, 12, 33, 31, 25,  8, 29, 41, 16, 28,
        16, 10, 23, 30, 31, 34,  1, 17,  0, 25, 38, 15, 12, 38])

# Forwarding process

In [21]:
target_encoding = model.target_encoder(target_images)

In [22]:
target_encoding.shape

torch.Size([32, 1664, 7, 7])

In [23]:
context_encoding = model.context_encoder(context_images)

In [24]:
context_encoding.shape

torch.Size([32, 1664, 7, 7])

In [25]:
context_encoding1, target_encoding1 = model.tokenizer(context_encoding, target_encoding)

In [26]:
context_encoding1.shape

torch.Size([49, 32, 1664])

In [27]:
target_encoding1.shape

torch.Size([1, 32, 1664])

In [28]:
context_encoding2, target_encoding2 = model.positional_encoding(context_encoding1, target_encoding1)

In [29]:
context_encoding2.shape

torch.Size([49, 32, 1664])

In [30]:
target_encoding2.shape

torch.Size([1, 32, 1664])

In [32]:
target_encoding3, attention_map = model.decoder(target_encoding2, context_encoding2)

In [33]:
target_encoding3.shape

torch.Size([1, 32, 1664])

In [34]:
attention_map.shape

torch.Size([32, 6, 1, 49])

In [28]:
torch.tensor([
    [[1,2]], [[1,2]], [[1,2]]
]).squeeze(1).shape

torch.Size([3, 2])

In [32]:
torch.tensor([
    [[1,2]], [[1,2]], [[1,2]]
])[:, 0].shape

torch.Size([3, 2])

In [15]:
bbox2token(model, bbox)

tensor([25, 22, 30, 39, 17, 20, 38, 41, 17, 36, 24, 40, 10,  7, 23, 24, 18, 16,
        28, 21, 22, 28, 26, 32, 24, 36,  5,  4, 34, 41, 12, 25],
       device='mps:0')

In [13]:
model.NUM_CONTEXT_TOKENS

49

In [ ]:
train.py
import os
import warnings
import argparse
import datetime
import pathlib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from tqdm import tqdm, trange

from test import test
from utils.evaluate_uncertainty import evaluate_uncertainty
from core.config import create_config, save_config
from core.dataset import COCODataset, COCODatasetWithID, COCODatasetGeneral
from core.model import Model
from core.metrics import AccuracyLogger


## Initialization
#
    
parser = argparse.ArgumentParser()
parser.add_argument("--config", type=str, help="Path to config file. If additional commandline options are provided, they are used to modify the specifications in the config file.")
parser.add_argument("--outdir", type=str, default="output/{date:%Y-%m-%d_%H%M}".format(date=datetime.datetime.now()), help="Path to output folder (will be created if it does not exist).")
parser.add_argument("--checkpoint", type=str, help="Path to model checkpoint from which to continue training.")
parser.add_argument("--annotations", type=str, help="Path to COCO-style annotations file.")
parser.add_argument("--imagedir", type=str, help="Path to images folder w.r.t. which filenames are specified in the annotations.")

parser.add_argument("--test_annotations", type=str, help="Path to COCO-style annotations file for model evaluation.")
parser.add_argument("--test_imagedir", type=str, help="Path to images folder w.r.t. which filenames are specified in the annotations for model evaluation.")
parser.add_argument("--test_frequency", type=int, default=1, help="Evaluate model on test data every __ epochs.")

parser.add_argument("--epochs", type=int, default=1, help="Number of epochs to train.")
parser.add_argument("--save_frequency", type=int, default=1, help="Save model checkpoint every __ epochs.")
parser.add_argument("--print_batch_metrics", action='store_true', default=False, help="Set to print metrics for every batch.")

parser.add_argument("--batch_size", type=int, help="Batchsize to use for training.")
parser.add_argument("--learning_rate", type=float, help="Learning rate to use for training.")
parser.add_argument("--imbalance_reweighting", action='store_true', help="Reweight samples in proportion to the number of samples per class.")
parser.add_argument("--num_decoder_heads", type=int, help="Number of decoder heads.")
parser.add_argument("--num_decoder_layers", type=int, help="Number of decoder layers.")
parser.add_argument("--uncertainty_gate_type", type=str, help="Uncertainty gating mechanism to use. Can be one of: 'entropy', 'relative_softmax_distance', 'learned', 'learned_metric'.")
parser.add_argument("--uncertainty_threshold", type=float, help="Uncertainty threshold for the uncertainty gating module. Note that training does not depend on the threshold, the model can still be used with different thresholds later.")
parser.add_argument("--weighted_prediction", action='store_true', default=None, help="If enabled, the model returns an uncertainty-weighted prediction if the uncertainty_gate prediction exceeds the uncertainty threshold.")
args = parser.parse_args()

# Create output directory
pathlib.Path(args.outdir).mkdir(exist_ok=True, parents=True)

# Load config or create a new one
cfg = create_config(args)

dataset = COCODatasetGeneral(cfg.annotations, cfg.imagedir, image_size =(224,224), normalize_means=[0.485, 0.456, 0.406], normalize_stds=[0.229, 0.224, 0.225])
dataloader = DataLoader(dataset, batch_size=cfg.batch_size, num_workers=4, shuffle=True, pin_memory=True, drop_last=True)

NUM_CLASSES = dataset.NUM_CLASSES
cfg.num_classes = NUM_CLASSES
save_config(cfg, args.outdir)
print(cfg)

model = Model.from_config(cfg)

assert(model.TARGET_IMAGE_SIZE == model.CONTEXT_IMAGE_SIZE == dataset.image_size), "Image size from the dataset is not compatible with the encoder."

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)

if cfg.imbalance_reweighting:
    class_weights = torch.true_divide(dataset.relative_annotation_counts.max(), dataset.relative_annotation_counts)
    criterion = nn.CrossEntropyLoss(weight= class_weights.to(device))
else:
    criterion = nn.CrossEntropyLoss()

if cfg.checkpoint is not None:
    print("Initializing from checkpoint {}".format(cfg.checkpoint))
    checkpoint = torch.load(cfg.checkpoint, map_location="cpu")
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
else:
    print("No checkpoint was passed.")
    model.to(device)
    start_epoch = 1

# Tensorboard
writer = SummaryWriter(log_dir=os.path.join(args.outdir, "runs/{date:%Y-%m-%d_%H%M}".format(date=datetime.datetime.now())))
context_images, target_images, bbox, labels = iter(dataloader).next()
writer.add_images("context_image_batch", context_images) # add example context image batch to tensorboard log
writer.add_images("target_image_batch", target_images) # add example target image batch to tensorboard log
with warnings.catch_warnings(): # add_graph method is known to issue a warning
    warnings.simplefilter("ignore")
    writer.add_graph(model, input_to_model=[context_images.to(device), target_images.to(device), bbox.to(device)]) # add model graph to tensorboard log

accuracy_logger_main_branch = AccuracyLogger(dataset.idx2label)
# accuracy_logger_uncertainty_branch = AccuracyLogger(dataset.idx2label)


## Training
#

for epoch in tqdm(range(start_epoch, args.epochs + 1), position=0, desc="Epochs", leave=True):

    model.train() # set train mode
    accuracy_logger_main_branch.reset() # reset accuracy logger every epoch
    # accuracy_logger_uncertainty_branch.reset()

    for i, (context_images, target_images, bbox, labels_cpu) in enumerate(tqdm(dataloader, position=1, desc="Batches", leave=True)):
        context_images = context_images.to(device)
        target_images = target_images.to(device)
        bbox = bbox.to(device)
        labels = labels_cpu.to(device) # keep a copy of labels on cpu to avoid unnecessary transfer back to cpu later

        # output_uncertainty_branch , output_main_branch, output_weighted, uncertainty = model(context_images, target_images, bbox)
        output_main_branch = model(context_images, target_images, bbox)

        # backpropagation through both branches
        optimizer.zero_grad(set_to_none=True)

        # if cfg.uncertainty_gate_type == "learned" or cfg.uncertainty_gate_type == "learned_metric":
        #     loss_uncertainty_estimator = criterion(output_weighted, labels)
        #     loss_uncertainty_estimator.backward(retain_graph=True)    

        # loss_uncertainty_branch = criterion(output_uncertainty_branch, labels)
        # loss_uncertainty_branch.backward(retain_graph=True)

        loss_main_branch = criterion(output_main_branch, labels)
        loss_main_branch.backward()

        optimizer.step()
        
        # log metrics
        # _, predictions_uncertainty_branch = torch.max(output_uncertainty_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        # batch_accuracy_uncertainty_branch = sum(predictions_uncertainty_branch == labels_cpu) / cfg.batch_size
        # batch_loss_uncertainty_branch = loss_uncertainty_branch.item()
        # writer.add_scalar("Batch Accuracy Uncertainty Branch/train", batch_accuracy_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # writer.add_scalar("Batch Loss Uncertainty Branch/train", batch_loss_uncertainty_branch, i + (epoch - 1) * len(dataloader))
        # accuracy_logger_uncertainty_branch.update(predictions_uncertainty_branch, labels_cpu)

        _, predictions_main_branch = torch.max(output_main_branch.detach().to("cpu"), 1) # choose idx with maximum score as prediction
        batch_accuracy_main_branch = sum(predictions_main_branch == labels_cpu) / cfg.batch_size
        batch_loss_main_branch = loss_main_branch.item()
        writer.add_scalar("Batch Accuracy Main Branch/train", batch_accuracy_main_branch, i + (epoch - 1) * len(dataloader))
        writer.add_scalar("Batch Loss Main Branch/train", batch_loss_main_branch, i + (epoch - 1) * len(dataloader))
        accuracy_logger_main_branch.update(predictions_main_branch, labels_cpu)

        # writer.add_scalar("Batch Uncertainty/train", torch.mean(uncertainty), i + (epoch - 1) * len(dataloader))

        if args.print_batch_metrics:
            print("\t Epoch {}, Batch {}: \t Loss: {} \t Accuracy: {}".format(epoch, i, batch_loss_main_branch, batch_accuracy_main_branch))


    # log metrics
    writer.add_scalar("Total Accuracy Main Branch/train", accuracy_logger_main_branch.accuracy(), epoch * len(dataloader))
    # writer.add_scalar("Total Accuracy Uncertainty Branch/train", accuracy_logger_uncertainty_branch.accuracy(), epoch * len(dataloader))

    print("\nEpoch {}, Train Accuracy: {}".format(epoch, accuracy_logger_main_branch.accuracy()))
    print("{0:20} {1:10}".format("Class", "Accuracy")) # header
    for name, acc in accuracy_logger_main_branch.named_class_accuarcies().items():
        writer.add_scalar("Class Accuracies Main Branch/train/{}".format(name), acc, epoch * len(dataloader))
        print("{0:20} {1:10.4f}".format(name, acc))

    # for name, acc in accuracy_logger_uncertainty_branch.named_class_accuarcies().items():
    #     writer.add_scalar("Class Accuracies Uncertainty Branch/train/{}".format(name), acc, epoch * len(dataloader))

    # save checkpoint and training accuracies
    if epoch % args.save_frequency == 0:
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict()}, args.outdir + "/checkpoint_{}.tar".format(epoch))
        print("Checkpoint saved.")

        accuracy_logger_main_branch.save(args.outdir, name="train_accuracies_epoch_{}".format(epoch))
        # accuracy_logger_uncertainty_branch.save(args.outdir, name="train_accuracies_uncertainty_branch_epoch_{}".format(epoch))
    
    # evaluation on test data
    if cfg.test_annotations is not None and cfg.test_imagedir is not None and epoch % args.test_frequency == 0:
        print("Starting evaluation on test data.")
        test_accuracy = test(model, cfg.test_annotations, cfg.test_imagedir, outdir=args.outdir, epoch=epoch)

        writer.add_scalar("Total Accuracy/test", test_accuracy.accuracy(), epoch * len(dataloader))
        for name, acc in test_accuracy.named_class_accuarcies().items():
            writer.add_scalar("Class Accuracies/test/{}".format(name), acc, epoch * len(dataloader))

        # print("Starting uncertainty evaluation.")
        # test_uncertainty_log = evaluate_uncertainty(model, cfg.test_annotations, cfg.test_imagedir)
        # writer.add_figure("Uncertainty Threshold Curve", test_uncertainty_log.plot_accuracy_vs_threshold(), epoch * len(dataloader))

        # if (args.epochs - epoch) / args.test_frequency < 1: # last evaluation
        #     writer.add_hparams({"learning_rate": cfg.learning_rate, "num_decoder_layers": cfg.num_decoder_layers, "num_decoder_heads": cfg.num_decoder_heads,
        #                         "uncertainty_gate_type": cfg.uncertainty_gate_type, "uncertainty_threshold": cfg.uncertainty_threshold, "imbalance_reweighting": str(cfg.imbalance_reweighting)},
        #                         metric_dict={"hparam/accuracy": test_accuracy.accuracy()})
        
writer.close()